# Lamoda · вероятность повторной покупки
Прогнозируем хотя бы одну покупку за три месяца после даты среза. Входные файлы можно подключить в Kaggle, распаковать в папку `data` или положить исходный ZIP рядом с ноутбуком. Для другого пути задайте переменную `LAMODA_DATA` (каталог или ZIP). Нужны pandas, NumPy, PyArrow и scikit-learn. Последняя ячейка записывает `submission.csv` в рабочий каталог.

In [1]:
from pathlib import Path
import os
from zipfile import ZipFile

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits

slug = "lamoda-hse-repeat-purchase-2026"
files = ["train.csv", "test.csv", "orders.parquet", "products.parquet", "sample_submission.csv"]
candidates = [Path("data"), Path("."), Path(slug + ".zip"),
              Path("/kaggle/input") / slug, Path("/kaggle/input/competitions") / slug]
attachments = Path("/workspace/attachments")
if attachments.exists():
    candidates.extend(sorted(attachments.glob("*/" + slug + ".zip")))
if "LAMODA_DATA" in os.environ:
    candidates = [Path(os.environ["LAMODA_DATA"])]
data = next((p for p in candidates if (p.is_file() and p.suffix == ".zip")
             or all((p / name).is_file() for name in files)), None)
if data is None:
    raise FileNotFoundError("Укажите путь к данным в LAMODA_DATA.")

def read_table(name, **kwargs):
    reader = pd.read_csv if name.endswith(".csv") else pd.read_parquet
    if data.is_dir():
        return reader(data / name, **kwargs)
    with ZipFile(data) as archive, archive.open(name) as stream:
        return reader(stream, **kwargs)

train = read_table("train.csv", parse_dates=["scoring_date"])
test = read_table("test.csv", parse_dates=["scoring_date"])
sample = read_table("sample_submission.csv")
orders = read_table("orders.parquet")
products = read_table("products.parquet", columns=["product_id", "brand", "category"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders = orders.merge(products, how="left", on="product_id", validate="many_to_one")
assert train.target.isin([0, 1]).all() and test.customer_id.is_unique
print(train.groupby("scoring_date").target.agg(["size", "mean"]).to_string())

               size     mean
scoring_date                
2022-01-01    25000  0.34764
2022-02-01    25000  0.33368
2022-03-01    25000  0.33924


### История клиента на дату прогноза
Строка истории соответствует товарной позиции. Считаем количество отдельных заказов, интервалы между днями покупок, суммы, скидки и разнообразие ассортимента. Окна в 30, 90, 180 и 365 дней отражают изменения активности. Там, где история отсутствует, бустинг сам обрабатывает пропуски.

In [2]:
def features(cohort):
    blocks = []
    for cutoff, customers in cohort.groupby("scoring_date", sort=False):
        history = orders.loc[
            orders.customer_id.isin(customers.customer_id) & (orders.order_date < cutoff)
        ].sort_values(["order_date", "order_id"], kind="stable")
        by = history.groupby("customer_id")
        stats = by.agg(
            first=("order_date", "min"), last=("order_date", "max"),
            n_orders=("order_id", "nunique"), n_days=("order_date", "nunique"),
            n_items=("product_id", "size"), n_brands=("brand", "nunique"),
            n_categories=("category", "nunique"), total_paid=("paid_price", "sum"),
            mean_price=("paid_price", "mean"), mean_discount=("discount_percent", "mean"),
            mean_loyalty=("loyalty_percent", "mean"), last_loyalty=("loyalty_percent", "last"),
        )
        stats["recency"] = (cutoff - stats.pop("last")).dt.days
        stats["age_days"] = (cutoff - stats.pop("first")).dt.days
        stats["mean_basket"] = stats.total_paid / stats.n_orders
        days = history[["customer_id", "order_date"]].drop_duplicates()
        gaps = days.groupby("customer_id").order_date.diff().dt.days
        stats = stats.join(gaps.groupby(days.customer_id).agg(["mean", "std", "last"]).add_prefix("gap_"))
        stats["recency_over_gap"] = stats.recency / stats.gap_mean
        for window in (30, 90, 180, 365):
            recent = history.loc[history.order_date >= cutoff - pd.Timedelta(days=window)]
            totals = recent.groupby("customer_id").agg(
                orders=("order_id", "nunique"), spend=("paid_price", "sum"),
                items=("product_id", "size"),
            )
            stats = stats.join(totals.reindex(stats.index, fill_value=0).add_suffix(f"_{window}"))
        block = stats.reindex(customers.customer_id)
        block.index = customers.index
        blocks.append(block)
    return pd.concat(blocks).reindex(cohort.index)

X = features(train)
X_test = features(test).loc[:, X.columns]
y = train.target
print("Матрицы признаков:", X.shape, X_test.shape)

Матрицы признаков: (75000, 29) (25000, 29)


### Проверка на последней размеченной когорте
Учимся на датах 1 января и 1 февраля, оцениваем ранжирование на 1 марта. Трёхмесячные окна целей перекрываются по календарю, поэтому эта проверка измеряет перенос между когортами. В качестве ориентира используем давность последней покупки. Для локальной AP@20% суммируем precision на позициях покупателей в верхней пятой части и делим на минимум из числа покупателей и длины этой части.

In [3]:
def ap20(labels, scores):
    k = max(1, len(labels) // 5)
    ranked = np.asarray(labels)[np.argsort(-np.asarray(scores), kind="stable")[:k]]
    precision = ranked.cumsum() / np.arange(1, k + 1)
    return (precision * ranked).sum() / max(1, min(k, labels.sum()))

def fit_model(features, labels):
    model = HistGradientBoostingClassifier(max_iter=180, learning_rate=0.055,
            max_leaf_nodes=23, min_samples_leaf=60, l2_regularization=3,
            early_stopping=False, random_state=42)
    with threadpool_limits(limits=4):
        model.fit(features, labels)
    return model

valid = train.scoring_date.eq(train.scoring_date.max())
model = fit_model(X.loc[~valid], y.loc[~valid])
with threadpool_limits(limits=4):
    valid_scores = model.predict_proba(X.loc[valid])[:, 1]
print(f"Давность покупки: {ap20(y[valid], -X.loc[valid, 'recency'].fillna(np.inf)):.5f}")
print(f"Градиентный бустинг: {ap20(y[valid], valid_scores):.5f}")

Давность покупки: 0.27979
Градиентный бустинг: 0.64908


### Прогноз на 1 апреля
Переобучаем модель на всех предоставленных метках. Собираем посылку в порядке строк образца Kaggle и проверяем сохранённый файл.

In [4]:
final_model = fit_model(X, y)
with threadpool_limits(limits=4):
    probabilities = final_model.predict_proba(X_test)[:, 1]
assert list(sample.columns) == ["customer_id", "score"]
assert sample.customer_id.is_unique and set(sample.customer_id) == set(test.customer_id)
scores = pd.Series(probabilities, index=test.customer_id)
submission = sample[["customer_id"]].copy()
submission["score"] = submission.customer_id.map(scores)
assert submission.score.between(0, 1).all()
submission.to_csv("submission.csv", index=False)
saved = pd.read_csv("submission.csv")
pd.testing.assert_frame_equal(saved, submission)
print(f"submission.csv · {len(saved):,} клиентов · средний score {saved.score.mean():.4f}")
print(saved.head().to_string(index=False))

submission.csv · 25,000 клиентов · средний score 0.3452
 customer_id    score
           7 0.484754
           8 0.186785
          12 0.356292
          13 0.064889
          18 0.683158
